
# Assignment 6 – Updated Page Rank Algorithm  
**Date:** 10-Feb-26  

NAME : Niveditha Anumandla

ROLL NO : U23CS011

In [1]:

# Required Libraries
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import requests
from bs4 import BeautifulSoup



## Step 1: Build Extended Web Graph of SVNIT
We first crawl the SVNIT homepage and then crawl one-hop neighbors.


In [2]:

def get_links(url):
    links = set()
    try:
        response = requests.get(url, timeout=5)
        soup = BeautifulSoup(response.text, 'html.parser')
        for a in soup.find_all('a', href=True):
            href = a['href']
            if href.startswith('http'):
                links.add(href)
    except:
        pass
    return links

seed_url = "https://www.svnit.ac.in"
G_svnit = nx.DiGraph()

first_hop = get_links(seed_url)
for link in first_hop:
    G_svnit.add_edge(seed_url, link)

# One-hop neighbors
for link in list(first_hop)[:10]:  # limit for safety
    second_hop = get_links(link)
    for l in second_hop:
        G_svnit.add_edge(link, l)

print("SVNIT Extended Graph Nodes:", G_svnit.number_of_nodes())
print("SVNIT Extended Graph Edges:", G_svnit.number_of_edges())


SVNIT Extended Graph Nodes: 275
SVNIT Extended Graph Edges: 499



## Step 2: Load Karate Club Graph


In [3]:

G_karate = nx.karate_club_graph()
print("Karate Graph Nodes:", G_karate.number_of_nodes())
print("Karate Graph Edges:", G_karate.number_of_edges())


Karate Graph Nodes: 34
Karate Graph Edges: 78



## Step 3: Simple PageRank Algorithm (Without Teleportation)


In [4]:

def simple_pagerank(G, iterations=100):
    N = len(G)
    pr = dict.fromkeys(G.nodes(), 1/N)
    for _ in range(iterations):
        new_pr = {}
        for node in G:
            rank_sum = 0
            for nbr in G.predecessors(node):
                rank_sum += pr[nbr] / G.out_degree(nbr)
            new_pr[node] = rank_sum
        pr = new_pr
    return pr



## Step 4: Updated PageRank with Teleportation (Damping Factor = 0.85)


In [5]:

def updated_pagerank(G, d=0.85, iterations=100):
    N = len(G)
    pr = dict.fromkeys(G.nodes(), 1/N)
    for _ in range(iterations):
        new_pr = {}
        for node in G:
            rank_sum = 0
            for nbr in G.predecessors(node):
                if G.out_degree(nbr) > 0:
                    rank_sum += pr[nbr] / G.out_degree(nbr)
            new_pr[node] = (1-d)/N + d * rank_sum
        pr = new_pr
    return pr



## Step 5: Apply PageRank Algorithms


In [8]:
# Karate Club Graph
G_karate_directed = G_karate.to_directed() # Convert to directed graph
simple_karate = simple_pagerank(G_karate_directed)
updated_karate = updated_pagerank(G_karate_directed)

# SVNIT Graph
simple_svnit = simple_pagerank(G_svnit)
updated_svnit = updated_pagerank(G_svnit)


## Step 6: Analysis of Results


In [9]:

def analyze(pr):
    max_node = max(pr, key=pr.get)
    min_node = min(pr, key=pr.get)
    return max_node, pr[max_node], pr[min_node]

print("Karate Club (Updated PageRank):")
print("Highest:", analyze(updated_karate))
print("Lowest PR:", min(updated_karate.values()))

print("\nSVNIT Graph (Updated PageRank):")
print("Highest:", analyze(updated_svnit))
print("Lowest PR:", min(updated_svnit.values()))


Karate Club (Updated PageRank):
Highest: (33, 0.10091918233262566, 0.009564745492135523)
Lowest PR: 0.009564745492135523

SVNIT Graph (Updated PageRank):
Highest: ('https://www.facebook.com/share/1CSGwKNaEC/', 0.0005646727776107749, 0.0005454545454545456)
Lowest PR: 0.0005454545454545456



## Step 7: Display Final PageRank Vectors


In [10]:

print("Final Updated PageRank Vector (Karate Club):")
for k, v in sorted(updated_karate.items(), key=lambda x: x[1], reverse=True):
    print(k, round(v, 5))

print("\nFinal Updated PageRank Vector (SVNIT Graph):")
for k, v in list(updated_svnit.items())[:10]:
    print(k, round(v, 5))


Final Updated PageRank Vector (Karate Club):
33 0.10092
0 0.097
32 0.07169
2 0.05708
1 0.05288
31 0.03716
3 0.03586
23 0.03152
8 0.02977
13 0.02954
5 0.02911
6 0.02911
29 0.02629
27 0.02564
30 0.02459
7 0.02449
4 0.02198
10 0.02198
24 0.02108
25 0.02101
19 0.0196
28 0.01957
16 0.01678
26 0.01504
12 0.01464
17 0.01456
21 0.01456
14 0.01454
15 0.01454
18 0.01454
20 0.01454
22 0.01454
9 0.01431
11 0.00956

Final Updated PageRank Vector (SVNIT Graph):
https://www.svnit.ac.in 0.00055
https://www.svnit.ac.in/Data/Notice/2025/November/Adobe Scan 14 Nov 2025 (1).pdf 0.00055
https://www.facebook.com/share/1CSGwKNaEC/ 0.00056
https://www.svnit.ac.in/Data/Notice/2025/May/Extension Notice.pdf 0.00055
https://www.svnit.ac.in/Data/Notice/2025/December/Brochure 3rd IC2S2TD-2026.pdf 0.00055
https://www.svnit.ac.in/web/annual-account-report.php 0.00056
https://www.svnit.ac.in/web/academic_section.php 0.00056
https://www.svnit.ac.in/web/results/result_april-may-2025.php 0.00055
https://www.svnit.ac.in/D


## Step 8: Comparison
- Updated PageRank avoids spider traps using teleportation.
- Rankings are more stable than Simple PageRank.
- Important hub nodes receive higher rank.



## Conclusion
Updated PageRank with teleportation produces more realistic rankings and avoids rank sinks,
making it suitable for real-world web graphs.
